# Ajuste fino de Strands Decider 2B con BANKING77

Entrena el decider con los ejemplos de entrenamiento de BANKING77 y lo evalúa con las mismas **3 080 consultas de prueba**, para compararlo con el decider original y con BERT.

Pasos: preparar datos → prueba corta de 20 pasos → entrenamiento → calibración → servidor local → evaluación.

**GPU: necesita 24 GB o más.** La T4 gratuita (16 GB) **no alcanza**. Usa **L4** o **A100** (Colab Pro): *Entorno de ejecución → Cambiar tipo de entorno de ejecución*.

Tiempo aproximado: unos 514 pasos de entrenamiento. En una L4, el repositorio oficial midió unos 0,22 pasos por segundo, es decir unos 40 minutos. En una A100 es más rápido. La calibración y la evaluación suman unos 10 a 15 minutos.

Usa el código de entrenamiento oficial de [strands-labs/strands-decider](https://github.com/strands-labs/strands-decider), fijado a una versión concreta.

## 1. Parámetros

In [ ]:
REPO_URL = "https://github.com/juan293114/strands-decider-2b"
DECIDER_URL = "https://github.com/strands-labs/strands-decider"
DECIDER_COMMIT = "b81d443c07438dc5ed51d0e85f9db184ad65d809"  # versión con la que se preparó este cuaderno

INIT_FROM = "StrandsAgents/strands-decider-2B-hobson-v21"  # el mismo modelo que sirve LiteLLM
EPOCAS = 2
CKPT = "/content/ckpt/s0"  # dónde se guardará el modelo ajustado

## 2. Comprobar la GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No hay GPU. Elige L4 o A100 en el tipo de entorno de ejecución."
nombre = torch.cuda.get_device_name(0)
memoria = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {nombre}, {memoria:.0f} GB")
assert memoria >= 22, "Hace falta una GPU de 24 GB o más (L4 o A100). La T4 no alcanza."
print("PyTorch", torch.__version__, "(el repositorio oficial necesita 2.7 o superior)")

## 3. Descargar los dos repositorios

In [ ]:
import os

%cd /content
if not os.path.exists("strands-decider-2b"):
    !git clone -q {REPO_URL} strands-decider-2b
if not os.path.exists("strands-decider"):
    !git clone -q {DECIDER_URL} strands-decider
!git -C strands-decider checkout -q {DECIDER_COMMIT}

## 4. Instalar dependencias

Son las versiones que indica el repositorio oficial. `flash-linear-attention` es obligatorio para el modelo Qwen3.5 en que se basa el decider.

Si PyTorch es menor que 2.7, ejecuta antes:
`!pip install torch==2.7.1 --index-url https://download.pytorch.org/whl/cu126` y reinicia la sesión.

In [ ]:
%cd /content/strands-decider
!pip install -q "transformers==5.17.0" "peft==0.21.0" flash-linear-attention
!pip install -q -e ".[train]"
%cd /content/strands-decider-2b
!pip install -q -r requirements.txt

## 5. Preparar los datos de entrenamiento

Convierte BANKING77 al formato oficial: 8 463 filas para entrenar y 1 540 reservadas para calibrar. Cada fila usa la misma instrucción y las mismas 77 opciones que el experimento.

In [ ]:
%cd /content/strands-decider-2b
!python 1_download_data.py
!python 5_prepare_decider_finetune.py --init-from {INIT_FROM} --epocas {EPOCAS} --checkpoint {CKPT}
!cat work/decider/train.yaml

## 6. Prueba corta de 20 pasos

Antes de entrenar, comprueba en la salida:

1. Aparece `continuing from StrandsAgents/...`. Si solo dice `loading base model`, no está partiendo del decider publicado.
2. `trainable N / M (P%)` con P de al menos 1 %.
3. `train=` y `val=` suman unas 8 463 filas.
4. La pérdida baja durante los 20 pasos.

Si algo falla, detente aquí.

In [ ]:
%cd /content/strands-decider
!python -m training.train --config /content/strands-decider-2b/work/decider/train.yaml --output-dir /content/ckpt/smoke --max-steps 20

## 7. Entrenamiento completo

In [ ]:
%cd /content/strands-decider
!python -m training.train --config /content/strands-decider-2b/work/decider/train.yaml

## 8. Calibrar la confianza

Ajusta la temperatura del modelo con las filas reservadas. No cambia qué intención elige, solo qué tan fiable es su probabilidad.

In [ ]:
%cd /content/strands-decider
!python -m training.evaluate calibrate {CKPT} --data /content/strands-decider-2b/work/decider/calib.jsonl --limit 1000000

## 9. Levantar el modelo ajustado como servidor local

In [ ]:
import subprocess, time, urllib.request

servidor = subprocess.Popen(
    ["strands-decider", "serve", CKPT, "--device", "cuda", "--port", "8000"],
    stdout=open("/content/serve.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(300):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5)
        print("Servidor listo")
        break
    except Exception:
        if servidor.poll() is not None:
            print(open("/content/serve.log").read()[-3000:])
            raise RuntimeError("El servidor se detuvo; revisa el log de arriba")
        time.sleep(2)
else:
    raise RuntimeError("El servidor no respondió en 10 minutos")

## 10. Evaluar con las 3 080 consultas

Envía cada consulta al modelo ajustado, igual que el experimento original, y guarda las respuestas en `results/decider_finetuned/names.jsonl`.

In [ ]:
%cd /content/strands-decider-2b
!python 2_run_experiment.py --por-clase 0 --condicion names --servidor-local http://127.0.0.1:8000

## 11. Comparar con el decider original y el resto de modelos

In [ ]:
!python 3_analyze_results.py
servidor.terminate()

## 12. Descargar los resultados

Descarga `resultados_decider_finetuned.zip`, que incluye las predicciones y la configuración del entrenamiento. Sube el contenido a `results/decider_finetuned/` en GitHub o envíaselo a Claude para actualizar el reporte.

In [ ]:
import glob, shutil

carpeta = "results/decider_finetuned"
for archivo in ["train_config.json", "history.json", "strands_decider_config.json"]:
    for ruta in glob.glob(f"{CKPT}/{archivo}"):
        shutil.copy(ruta, carpeta)
zip_path = shutil.make_archive("resultados_decider_finetuned", "zip", "results", "decider_finetuned")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Fuera de Colab. El archivo está en:", zip_path)

## Opcional: guardar el modelo ajustado en Google Drive

La sesión de Colab se borra al cerrarla. Si quieres conservar el modelo para servirlo después, cópialo a Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!cp -r {CKPT} /content/drive/MyDrive/strands-decider-banking77